# GRPO with OpenReward (Local ORS) + Toolathlon Gym + Qwen3.5-4B

This notebook demonstrates end-to-end GRPO training with:
- `trl.experimental.openreward.OpenRewardSpec`
- a **local ORS server started from inside this notebook**
- Toolathlon Gym environment
- [`Qwen/Qwen3.5-4B`](https://huggingface.co/qwen/Qwen3.5-4B) via standard `transformers` + `peft` LoRA

> ⚠️ This notebook is intended for **local Jupyter** environments with Docker / enough system access. Qwen3.5 needs **Transformers v5** (see the guide).

## 1) Install dependencies

In [1]:
# If running in a fresh environment, uncomment and run.
# You may need to restart kernel/runtime after install.

# !pip install -U "trl[openreward]" "transformers>=5.2" peft openreward accelerate bitsandbytes

## 2) Optional: Hugging Face login

In [2]:
# from huggingface_hub import notebook_login
# notebook_login()

## 3) Configure local ORS startup (Jupyter-managed)

This notebook defaults to the prebuilt container image:
- `ghcr.io/rycerzes/toolathlon-gym:latest`

If the package is private, authenticate to GHCR in a cell before startup:
```bash
# !echo $GHCR_PAT | docker login ghcr.io -u <github-username> --password-stdin
```

You can still override startup command for a custom local setup.


In [3]:
import shlex

# Prebuilt Toolathlon Gym image from GHCR.
TOOLATHLON_IMAGE = "ghcr.io/rycerzes/toolathlon-gym:latest"
CONTAINER_NAME = "toolathlon-ors"
HOST_PORT = 8080
CONTAINER_PORT = 8080

# Set True if you want to pull latest on each run.
DOCKER_PULL_BEFORE_START = True

# Default startup command: detached container mode.
ORS_BOOTSTRAP_CMD = (
    f"docker run -d --name {CONTAINER_NAME} -p {HOST_PORT}:{CONTAINER_PORT} {TOOLATHLON_IMAGE}"
)

# Local URL used by OpenReward SDK + TRL OpenRewardSpec.
URL = f"http://127.0.0.1:{HOST_PORT}"

# ORS environment name exposed by the server.
ENV_NAME = "toolathlongym"

# Local server implementation currently exposes the train split.
SPLIT = "train"

# Small slice: fewer ORS probe sessions at OpenRewardSpec() time + faster epochs.
NUM_TASKS = 4


## 4) Start local ORS server from notebook

In [4]:
import atexit
import subprocess
import shlex

def _run(cmd: str, check: bool = True):
    p = subprocess.run(
        shlex.split(cmd),
        capture_output=True,
        text=True,
    )
    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed: {cmd}\n"
            f"exit={p.returncode}\nstdout=\n{p.stdout}\nstderr=\n{p.stderr}"
        )
    return p

def _get_running_container_id(name: str) -> str:
    p = _run(f"docker ps --filter name=^/{name}$ --format {{.ID}}", check=False)
    return p.stdout.strip()

running_id = _get_running_container_id(CONTAINER_NAME)
if running_id:
    print(f"ORS container already running: {CONTAINER_NAME} ({running_id})")
    container_id = running_id
else:
    # Remove stale stopped container if present.
    _run(f"docker rm -f {CONTAINER_NAME}", check=False)

    if DOCKER_PULL_BEFORE_START:
        print(f"Pulling image: {TOOLATHLON_IMAGE}")
        _run(f"docker pull {TOOLATHLON_IMAGE}")

    print(f"Starting ORS container with: {ORS_BOOTSTRAP_CMD}")
    start = _run(ORS_BOOTSTRAP_CMD)
    container_id = start.stdout.strip()

    if not container_id:
        raise RuntimeError("docker run -d returned empty container id")

    print(f"Started ORS container: {CONTAINER_NAME} ({container_id})")

def _cleanup_ors_container():
    _run(f"docker rm -f {CONTAINER_NAME}", check=False)

atexit.register(_cleanup_ors_container)
globals()["_ors_container_id"] = container_id
print(f"URL: {URL}")


ORS container already running: toolathlon-ors ({.ID})
URL: http://127.0.0.1:8080


## 5) Health-check local ORS

In [5]:
import time
import requests
import subprocess
import shlex

def _run(cmd: str, check: bool = True):
    p = subprocess.run(shlex.split(cmd), capture_output=True, text=True)
    if check and p.returncode != 0:
        raise RuntimeError(
            f"Command failed: {cmd}\n"
            f"exit={p.returncode}\nstdout=\n{p.stdout}\nstderr=\n{p.stderr}"
        )
    return p

def _running_container_id(name: str) -> str:
    p = _run(f"docker ps --filter name=^/{name}$ --format {{.ID}}", check=False)
    return p.stdout.strip()

health_url = f"{URL}/health"
timeout_s = 180
start_t = time.time()

ready = False
last_err = None
while time.time() - start_t < timeout_s:
    cid = _running_container_id(CONTAINER_NAME)
    if not cid:
        logs = _run(f"docker logs --tail 200 {CONTAINER_NAME}", check=False)
        raise RuntimeError(
            f"Container {CONTAINER_NAME} is not running.\n"
            f"docker logs stdout/stderr:\n{logs.stdout}\n{logs.stderr}"
        )

    try:
        r = requests.get(health_url, timeout=2)
        if r.status_code == 200:
            ready = True
            break
    except Exception as e:
        last_err = e

    time.sleep(1)

if not ready:
    logs = _run(f"docker logs --tail 200 {CONTAINER_NAME}", check=False)
    raise TimeoutError(
        f"ORS health check did not pass within {timeout_s}s. Last error={last_err}.\n"
        f"docker logs stdout/stderr:\n{logs.stdout}\n{logs.stderr}"
    )

print(f"ORS is healthy at {health_url}")


ORS is healthy at http://127.0.0.1:8080/health


## 6) (Optional) Tail local ORS logs

In [6]:
import subprocess
import shlex

def _run(cmd: str):
    p = subprocess.run(shlex.split(cmd), capture_output=True, text=True)
    print(f"$ {cmd}")
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)

_run(f"docker ps --filter name=^/{CONTAINER_NAME}$")
_run(f"docker logs --tail 200 {CONTAINER_NAME}")


$ docker ps --filter name=^/toolathlon-ors$
CONTAINER ID   IMAGE                                    COMMAND                CREATED       STATUS       PORTS                                         NAMES
38b4f124a781   ghcr.io/rycerzes/toolathlon-gym:latest   "/app/entrypoint.sh"   2 hours ago   Up 2 hours   0.0.0.0:8080->8080/tcp, [::]:8080->8080/tcp   toolathlon-ors

$ docker logs --tail 200 toolathlon-ors
2026-05-11T16:25:06.657254Z [info     ] request_handled                [openreward.environments.server] httpRequest={'latency': '0.000230s', 'status': 200} method=POST path=/ping session_id=a174aba2-40fb-44a4-acad-72ba57c145cb
2026-05-11T16:25:15.142044Z [info     ] request_handled                [openreward.environments.server] httpRequest={'latency': '0.000233s', 'status': 200} method=POST path=/ping session_id=fa633611-4592-4bae-b980-dd332b42350a
2026-05-11T16:25:16.658571Z [info     ] request_handled                [openreward.environments.server] httpRequest={'latency': '0.00012

## 7) Configure single-host OpenReward overrides

Required for local/single-host servers so SDK does not expect split subdomains.

In [7]:
import os

os.environ["OPENREWARD_API_URL"] = URL
os.environ["OPENREWARD_SESSION_URL"] = URL

# os.environ["OPENREWARD_API_KEY"] = "..."

print("OPENREWARD_API_URL:", os.environ["OPENREWARD_API_URL"])
print("OPENREWARD_SESSION_URL:", os.environ["OPENREWARD_SESSION_URL"])

OPENREWARD_API_URL: http://127.0.0.1:8080
OPENREWARD_SESSION_URL: http://127.0.0.1:8080


## 8) Smoke test SDK against local ORS

In [8]:
from openreward import OpenReward

or_client = OpenReward(base_url=URL)
environment = or_client.environments.get(name=ENV_NAME, base_url=URL)

print("Splits:", environment.list_splits())
print(f"num_tasks({SPLIT}):", environment.num_tasks(SPLIT))

tools = environment.list_tools()
print("Tool count:", len(tools))
print("First tools:", [t.name if hasattr(t, "name") else str(t) for t in tools[:10]])

Splits: ['train']
num_tasks(train): 503
Tool count: 2
First tools: ['claim_done', 'python_execute']


## 9) Build OpenRewardSpec

[Toolathlon Gym](https://github.com/rycerzes/toolathlon-gym) has **per-task MCP tools** — each task needs a different subset of the 25 MCP servers. However, the environment handles this via two meta-tools (`get_tool_details`, `call_tool`) returned by [`list_task_tools()`](https://docs.openreward.ai/environments/using-task-specific-tools#using-task-specific-tools) that are **the same for every task**; the actual per-task tool catalog is inlined into the system prompt. So we only need to probe **one** task index for tool discovery.

We use `task_tools_discovery_index=0` to open a single discovery session (rather than one per task) and `num_tasks=NUM_TASKS` to control how many tasks appear in the dataset. This avoids redundant ORS sessions while still binding all four tools (`claim_done`, `python_execute`, `get_tool_details`, `call_tool`) for GRPO.


In [9]:
from trl.experimental.openreward import OpenRewardSpec

spec = OpenRewardSpec(
    URL,
    env_name=ENV_NAME,
    split=SPLIT,
    num_tasks=NUM_TASKS,
    task_tools_discovery_index=0,
)

ds = spec.train_dataset
print(ds)
print("Columns:", ds.column_names)
print("Sample row keys:", list(ds[0].keys()))


/tmp/ipykernel_481083/3581688352.py:1: TRLExperimentalWarning: You are importing from 'trl.experimental'. APIs here are unstable and may change or be removed without notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  from trl.experimental.openreward import OpenRewardSpec


Dataset({
    features: ['prompt', 'task_index', 'task_name'],
    num_rows: 4
})
Columns: ['prompt', 'task_index', 'task_name']
Sample row keys: ['prompt', 'task_index', 'task_name']


/home/asyin/swapnil/trl-openreward/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 10) Load Transformers model + LoRA (Qwen3.5-4B)

Uses standard `transformers` + `peft` LoRA with bf16 loading and tighter training settings to keep memory usage down without 4-bit QLoRA.

In [10]:
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer

max_seq_length = 4096
lora_rank = 16

model_name = "qwen/Qwen3.5-4B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
model.config.use_cache = False

peft_config = LoraConfig(
    r=lora_rank,
    lora_alpha=lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0,
    bias="none",
)
model = get_peft_model(model, peft_config)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Fetching 2 files: 100%|██████████| 2/2 [00:00<00:00, 12264.05it/s]
[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 426/426 [00:05<00:00, 83.30it/s] 


## 11) GRPO configuration

Toolathlon's **real** score is binary on **`claim_done`**, so sparse **0** outcome rewards are normal for a short run. For a **this demo** (moving **`reward`**, **`reward_std`**, **`Training Loss`**, **`kl`**), this notebook uses a **shaped** scalar: last ORS outcome plus a **tiny bonus per tool step** (`len(env.rewards)`). That is **not** the official benchmark reward — switch to `spec.reward_funcs` for strict Toolathlon semantics.

Also relaxes **completion / tool-loop limits** so multi-turn episodes are less likely to die early. Setting `beta=0.0` skips the reference-model copy, which is the main memory win for this non-Unsloth setup; restore a nonzero beta only if you have extra VRAM.

In [11]:
from trl import GRPOConfig


def demo_grpo_openreward_reward(environments, **kwargs):
    """ORS outcome (``env.reward``) + small per-tool-call bonus for visible GRPO metrics in short demos."""
    out = []
    for env in environments:
        outcome = float(env.reward)
        n_calls = len(env.rewards)
        out.append(outcome + 0.03 * min(n_calls, 30))
    return out


training_args = GRPOConfig(
    output_dir="outputs/grpo-openreward-toolathlon-qwen3_5-4b",
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_generations=2,
    max_completion_length=1024,
    max_tool_calling_iterations=32,
    max_steps=20,
    num_train_epochs=3,
    logging_steps=1,
    log_completions=False,
    log_unique_prompts=False,
    report_to="none",
    use_vllm=False,
    optim="adamw_torch",
    gradient_checkpointing=True,
    chat_template_kwargs={"enable_thinking": False},
    beta=0.0,
)

training_args

GRPOConfig(output_dir='outputs/grpo-openreward-toolathlon-qwen3_5-4b', per_device_train_batch_size=1, num_train_epochs=3, max_steps=20, learning_rate=1e-05, lr_scheduler_type=<SchedulerType.LINEAR: 'linear'>, lr_scheduler_kwargs=None, warmup_steps=0, optim=<OptimizerNames.ADAMW_TORCH: 'adamw_torch'>, optim_args=None, weight_decay=0.0, adam_beta1=0.9, adam_beta2=0.999, adam_epsilon=1e-08, optim_target_modules=None, gradient_accumulation_steps=2, average_tokens_across_devices=True, max_grad_norm=1.0, label_smoothing_factor=0.0, bf16=True, fp16=False, bf16_full_eval=False, fp16_full_eval=False, tf32=None, gradient_checkpointing=True, gradient_checkpointing_kwargs=None, torch_compile=False, torch_compile_backend=None, torch_compile_mode=None, use_liger_kernel=False, liger_kernel_config=None, use_cache=False, neftune_noise_alpha=None, torch_empty_cache_steps=None, auto_find_batch_size=False, logging_strategy=<IntervalStrategy.STEPS: 'steps'>, logging_steps=1, logging_first_step=False, log_o

## 12) Create trainer and train

Jupyter's built-in `NotebookProgressCallback` only renders "Training Loss" in the HTML table — all other GRPO metrics (reward, advantages, completion lengths, KL, etc.) are silently dropped. We swap it for the terminal `ProgressCallback` which prints the full metrics dict each logging step.

In [12]:
from trl import GRPOTrainer
from transformers import ProgressCallback
from transformers.utils.notebook import NotebookProgressCallback

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=spec.train_dataset,
    environment_factory=spec.environment_factory,
    reward_funcs=demo_grpo_openreward_reward,
)

# Swap the Jupyter callback for the terminal one so all GRPO metrics are printed.
trainer.remove_callback(NotebookProgressCallback)
trainer.add_callback(ProgressCallback)

train_stats = trainer.train()
train_stats


/tmp/ipykernel_481083/892956654.py:5: UserWarning: You are using 'environment_factory', which is an experimental feature. This API may change or be removed at any time without prior notice. Silence this warning by setting environment variable TRL_EXPERIMENTAL_SILENCE=1.
  trainer = GRPOTrainer(
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248044}.
  5%|▌         | 1/20 [01:11<22:39, 71.55s/it]

{'loss': '0', 'grad_norm': '0', 'learning_rate': '1e-05', 'num_tokens': '7994', 'completions/mean_length': '474.5', 'completions/min_length': '424', 'completions/max_length': '525', 'completions/clipped_ratio': '0.5', 'completions/mean_terminated_length': '424', 'completions/min_terminated_length': '424', 'completions/max_terminated_length': '424', 'tools/call_frequency': '2.5', 'tools/failure_frequency': '0.2', 'rewards/demo_grpo_openreward_reward/mean': '0.06', 'rewards/demo_grpo_openreward_reward/std': '0', 'reward': '0.06', 'reward_std': '0', 'frac_reward_zero_std': '1', 'entropy': '0.2237', 'clip_ratio/low_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/high_max': '0', 'clip_ratio/region_mean': '0', 'step_time': '71.43', 'epoch': '0.25'}


 10%|█         | 2/20 [02:36<23:49, 79.44s/it]

{'loss': '-0.03717', 'grad_norm': '0.2405', 'learning_rate': '9.5e-06', 'num_tokens': '1.678e+04', 'completions/mean_length': '494', 'completions/min_length': '468', 'completions/max_length': '520', 'completions/clipped_ratio': '0', 'completions/mean_terminated_length': '494', 'completions/min_terminated_length': '468', 'completions/max_terminated_length': '520', 'tools/call_frequency': '5.5', 'tools/failure_frequency': '0.2727', 'rewards/demo_grpo_openreward_reward/mean': '0.12', 'rewards/demo_grpo_openreward_reward/std': '0.08485', 'reward': '0.12', 'reward_std': '0.08485', 'frac_reward_zero_std': '0', 'entropy': '0.146', 'clip_ratio/low_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/high_max': '0', 'clip_ratio/region_mean': '0', 'step_time': '84.94', 'epoch': '0.5'}


 15%|█▌        | 3/20 [03:56<22:33, 79.64s/it]

...

100%|██████████| 20/20 [28:02<00:00, 85.72s/it]

{'loss': '-0.1591', 'grad_norm': '0.2312', 'learning_rate': '5e-07', 'num_tokens': '1.966e+05', 'completions/mean_length': '479', 'completions/min_length': '371', 'completions/max_length': '587', 'completions/clipped_ratio': '0.5', 'completions/mean_terminated_length': '587', 'completions/min_terminated_length': '587', 'completions/max_terminated_length': '587', 'tools/call_frequency': '5.5', 'tools/failure_frequency': '0.09091', 'rewards/demo_grpo_openreward_reward/mean': '0.15', 'rewards/demo_grpo_openreward_reward/std': '0.04243', 'reward': '0.15', 'reward_std': '0.04243', 'frac_reward_zero_std': '0', 'entropy': '0.1405', 'clip_ratio/low_mean': '0', 'clip_ratio/low_min': '0', 'clip_ratio/high_mean': '0', 'clip_ratio/high_max': '0', 'clip_ratio/region_mean': '0', 'step_time': '83.26', 'epoch': '5'}


100%|██████████| 20/20 [28:05<00:00, 84.27s/it]

{'train_runtime': '1685', 'train_samples_per_second': '0.024', 'train_steps_per_second': '0.012', 'train_loss': '-0.0388', 'epoch': '5'}


TrainOutput(global_step=20, training_loss=-0.038804607838392256, metrics={'train_runtime': 1685.3213, 'train_samples_per_second': 0.024, 'train_steps_per_second': 0.012, 'total_flos': 0.0, 'train_loss': -0.038804607838392256})

## 12b) Inspect training metrics

All GRPO metrics are stored in `trainer.state.log_history`. This cell extracts the key ones into a table.

In [13]:
import pandas as pd

history = trainer.state.log_history
# Filter to training steps (have 'loss' key)
rows = [h for h in history if "loss" in h]

key_cols = [
    "step", "loss", "reward", "reward_std",
    "completions/mean_length", "completions/clipped_ratio",
    "tools/call_frequency",
]
# Only keep columns that exist in the data.
available = [c for c in key_cols if any(c in r for r in rows)]
df = pd.DataFrame(rows)[available]
df


,step,loss,reward,reward_std,completions/mean_length,completions/clipped_ratio,tools/call_frequency
0,1,0.000000,0.060,0.000000,474.5,0.5,2.5
1,2,-0.037172,0.120,0.084853,494.0,0.0,5.5
2,3,-0.231897,0.165,0.063640,411.0,0.0,6.5
3,4,-0.000854,0.120,0.084853,413.5,0.0,10.0
4,5,-0.137718,0.225,0.106066,277.0,0.0,7.5
5,6,0.007134,0.090,0.084853,396.0,1.0,7.5
6,7,0.091744,0.105,0.021213,560.0,0.0,6.5
7,8,-0.298464,0.165,0.063640,417.5,0.0,6.0
8,9,-0.216313,0.150,0.042426,399.5,0.0,5.0
9,10,0.000000,0.150,0.000000,573.5,0.0,6.0


## 13) Optional: quick environment rollout sanity check

In [14]:
env = spec.environment_factory()
try:
    prompt_text = env.reset(**spec.train_dataset[0])
    print(prompt_text[:1200])
finally:
    env._close()

Accessible workspace directory: /tmp/workspaces/7924afbf-70f6-4155-a8fb-db4a39a2555a
When processing tasks, if you need to read/write local files and the user provides a relative path, you may choose to combine it with the above workspace directory to get the complete path.
If you believe the task is completed, you can either call the `local-claim_done` tool or respond without calling any tool to indicate completion. This will immediately terminate the task, and you will have no further opportunity to work on it.


## Tools

You have **two layers** of tools. Treat them differently.

### 1. Direct tools — call by name like any normal tool

Your top-level tool list contains a small set of tools registered directly by the environment, including `get_tool_details`, `call_tool`, `python_execute`, and the task-completion tool (check your tool list for its exact registered name, e.g. `claim_done`). Invoke any of these the way you invoke any function tool — by its own name, with its own argume

## 14) Save LoRA adapter

In [15]:
adapter_dir = "outputs/grpo-openreward-toolathlon-qwen3_5-4b/adapter"
trainer.save_model(adapter_dir)
print(f"Saved adapter to: {adapter_dir}")

Saved adapter to: outputs/grpo-openreward-toolathlon-qwen3_5-4b/adapter


## 15) Stop local ORS server (cleanup)

In [16]:
import subprocess
import shlex

def _run(cmd: str):
    p = subprocess.run(shlex.split(cmd), capture_output=True, text=True)
    print(f"$ {cmd}")
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)

_run(f"docker stop {CONTAINER_NAME}")
_run(f"docker rm -f {CONTAINER_NAME}")
globals()["_ors_container_id"] = None
print("ORS container cleanup completed.")


$ docker stop toolathlon-ors
toolathlon-ors

$ docker rm -f toolathlon-ors
toolathlon-ors

ORS container cleanup completed.
